In [ ]:
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt
import os 
import glob
from tqdm import tqdm
from src.config import intersection, lookup, diff
import pickle

### Define directories

In [ ]:
# Define directories
Base='/home/projects/cpr_man/people/lilniu/projects/target/data/data_genotype/subset/Formatted/plink_qc4/gemma/'
result_path = Base + 'results_final/'
figure_path = Base + 'analysis/figures/'
annotation_path = Base + 'analysis/annotation/'
crossref_path = Base + 'analysis/cross_reference'

### Check if any assoc files are missing

In [ ]:
phenotype_to_proteinID = pd.read_csv(os.path.join(Base,'ProteinID.txt'), sep='\t')

In [ ]:
# Read summary statistics after filtering for genome-wide significance (<5x10-8)
assoc_folder = 'assoc/'
os.chdir(os.path.join(result_path,assoc_folder))
assoc_files = glob.glob('*.txt')
assoc_phenotypes = [int(i.split('.')[0]) for i in assoc_files]
print('Numer of files: {}'.format(len(assoc_files)))

In [ ]:
missing_in_assoc = diff(phenotype_to_proteinID['Phenotype ID'].astype(int).tolist(), assoc_phenotypes)
print('Missing {} assoc files'.format(len(missing_in_assoc)))
print(missing_in_assoc)

### Read summary statistics

In [ ]:
# Read summary statistics after filtering for genome-wide significance (<5x10-8)
sig_folder = 'sig/'
os.chdir(os.path.join(result_path,sig_folder))
my_files = glob.glob('*.txt')
files = []
for i in tqdm(my_files):
    df = pd.read_csv(os.path.join(result_path,sig_folder,i), sep='\t').drop(['Unnamed: 0'], axis=1)
    files.append(df)
    
# Check if any file is missing
print('Numer of files: {}'.format(len(my_files)))
missing_files = [i for i in np.arange(1,1217) if i not in sorted([int(i.split('_')[0]) for i in my_files])]
print('Missing file:{}'.format(missing_files))

### Combine all significant associations

In [ ]:
# Combine all significant hits
df_sig = pd.concat(files)

# Check if any reference and alternative SNPs are flipped
df_sig['REF'] = df_sig['rs'].str.split('_').str[2]
df_sig['ALT'] = df_sig['rs'].str.split('_').str[3]
flipped = df_sig[df_sig['REF']!= df_sig['allele0']]
print('Reference and alternative SNPs were flipped in the following rows: {}'.format(flipped))

# Create two new columns of 'Protein ID' and 'Gene name' by splitting 'phenotype'
df_sig['Protein ID'] = df_sig['phenotype'].str.split('_').str[0]
df_sig['Gene name'] = df_sig['phenotype'].str.split('_').str[1]
df_sig['pqtl_id'] = df_sig['rs'].astype(str) + '_' + df_sig['phenotype'].astype(str)
df_sig.to_csv(os.path.join(Base, 'analysis/dataset/all_sig_pqtls.csv'), index=False)

In [ ]:
vcf_path ='/home/projects/cpr_man/people/lilniu/projects/target/data/data_genotype/subset/Formatted/plink_qc4'
with open (os.path.join(vcf_path, 'all_sig_pqtl_id.txt'), 'w') as f:
    for line in df_sig['rs'].unique().tolist():
        f.write(line + '\n')

In [ ]:
# Check inflation factor lambda
# script: https://github.com/pgxcentre/lambda/blob/master/compute_lambda.py
lambda_folder = 'output_final/lambda'
os.chdir(os.path.join(Base,lambda_folder))
my_files = glob.glob('*.txt')
files = []
for i in tqdm(my_files):
    df=pd.read_csv(os.path.join(Base, lambda_folder, i), sep=' ', header=None)
    files.append(df)
lambda_log = pd.concat(files)
lambda_log[3].describe()

### Export variants to be annotated by variant effect predictor

In [ ]:
# Prepare data for VEP annotation
target5_bim = pd.read_csv(os.path.join(Base,'target5.bim'), sep='\s+', header=None, 
            names=['CHR', 'SNP', 'dummy', 'POS', 'ALT', 'REF'])
target5_bim['allele']=target5_bim['REF']+'/'+target5_bim['ALT']
target5_bim['strand']='+'
target5_bim_sig = target5_bim[target5_bim['SNP'].isin(df_sig['rs'])]
vep_input = target5_bim_sig[['CHR', 'POS', 'POS', 'allele','strand', 'SNP']]
vep_input.to_csv(os.path.join(Base, 'target5.vep.input'), sep='\t', header=None, index=False)

In [ ]:
# https://grch37.ensembl.org/Homo_sapiens/Tools/VEP with RefSeq transcripts as reference 2024.03.02
# Read VEP annotation
vep_output = pd.read_csv(os.path.join(Base, 'target5.vep.output.txt'), sep='\t', na_values='-', low_memory=False)

# Create a dictionary that matches SNP to rsID
rsid = vep_output[['#Uploaded_variation', 'Existing_variation']].drop_duplicates()
IDmapping_snp_to_rs = dict(zip(rsid['#Uploaded_variation'], rsid['Existing_variation'].str.split(',').str[0]))

consequences = vep_output[['#Uploaded_variation', 'Consequence']].drop_duplicates()
# Create a dictionary that matches SNPs and consequences
dict_vep = {}
for i, group in consequences.groupby('#Uploaded_variation'):
    dict_vep[i] = group['Consequence'].tolist()

### Primary associations

#### Read clump results

- physical distance (±1 Mb) and LD threshold of r2 > 0.2, significance level 5e-8
- check script 
/home/projects/cpr_/people/lilniu/projects/target/data/data_genotype/subset/Formatted/plink_qc4/gemma/analysis/script_clump.sh

In [ ]:
# Create a dictionary that matches phenotype ID and Protein ID
phenotype_to_proteinID = pd.read_csv(os.path.join(Base,'ProteinID.txt'), sep='\t')
IDmapping_phenotype_to_proteinID = dict(zip(phenotype_to_proteinID['Phenotype ID'], phenotype_to_proteinID['Protein ID']))

In [ ]:
# Combine clumped results for all proteins
clump_path = result_path + 'assoc/clump_results/'
os.chdir(clump_path)
clump_files = glob.glob('*.clumped')

files = [pd.read_csv(file, sep='\s+') for file in clump_files]

for df, file in zip(files, clump_files):
    phenotype = int(file.split('.')[0])
    proteinID = IDmapping_phenotype_to_proteinID[phenotype]
    df['phenotype'] = proteinID
    
clumped_snps = pd.concat(files, ignore_index=True)
clumped_snps=clumped_snps.assign(pqtl_id=lambda x:x['SNP'].astype(str)+'_'+x['phenotype'].astype(str))

In [ ]:
# Clump HLA region (chr6: 29691116–33054976)
# For each protein, drop SNPs from the HLA region while keeping SNPs with the smallest p-value
from src.config import refine_snps_excluding_hla
clumped_refined = refine_snps_excluding_hla(clumped_snps)

In [ ]:
# Extract associations after clumping
df_sig_primary = lookup(df_sig, 'pqtl_id', clumped_refined['pqtl_id'].unique())
# Compute additional columns
df_sig_primary = df_sig_primary.assign(
    maf=lambda x: np.where(x['af'] < 0.5, x['af'], 1 - x['af']),
    beta_abs=lambda x: x['beta'].abs(),
    rsID=lambda x: x['rs'].map(IDmapping_snp_to_rs),
)

df_sig_primary.to_csv(os.path.join(Base,'analysis/dataset/df_sig_primary.csv'), index=False)

In [ ]:
df_sig_primary['phenotype'].nunique()

In [ ]:
df_sig_primary.shape

### Exclude artefactual pQTLs

In [ ]:
# Export primary pQTLs for peptide level assessment
df_sig_primary.to_csv(os.path.join(Base, 'analysis/alphamap/astral/primary_pqtls.csv'), index=False)
# Assessment done in target/data/data_genotype/subset/Formatted/plink_qc4/gemma/analysis/pqtl-peptide.ipynb
# Import final set with peptide level evidence 
final_set=pd.read_csv(os.path.join(Base, 'analysis/alphamap/astral/final_set.csv'))
df_sig_primary=lookup(df_sig_primary, 'pqtl_id', final_set['pqtl_id'].unique())

In [ ]:
vcf_path ='/home/projects/cpr_man/people/lilniu/projects/target/data/data_genotype/subset/Formatted/plink_qc4'
with open (os.path.join(vcf_path, 'all_sig_pqtl_id.txt'), 'w') as f:
    for line in df_sig['rs'].tolist():
        f.write(line + '\n')

#### Write to results

In [ ]:
with open (os.path.join(vcf_path, 'primary_pqtl_id.txt'), 'w') as f:
    for line in df_sig_primary['rs'].tolist():
        f.write(line + '\n')

### cis/trans pQTLs

In [ ]:
%run /home/projects/cpr_man/people/lilniu/projects/target/data/data_genotype/subset/Formatted/plink_qc4/gemma/analysis/TSS.ipynb

In [ ]:
# Import mapping file from Protein ID to Transcription start site - refer to notebook TSS.ipynb
df_mapping = pd.read_csv(os.path.join(Base, 'analysis/annotation/TSS_mapping_primary_pQTLs_20240127.csv'))
df_mapping.drop(['Gene name'], axis=1, inplace=True)

# Add columns of protein coding location
df_sig_primary_genomapped = df_sig_primary.merge(df_mapping, how='left', on='Protein ID')


# Categorize cis and trans pQTLs 
from src.config import cis_trans_pqtl
df_sig_primary_cistrans = cis_trans_pqtl(data=df_sig_primary_genomapped, 
                                  chr_snp='chr', 
                                  pos_snp='ps', 
                                  chr_prot='chr_protein', 
                                  pos_protein ='Transcription start site (TSS)')

# Drop proteins whose genomic coordinates cannot be mapped
df_sig_primary_cistrans.dropna(subset=['values'], inplace=True)
df_sig_primary_cistrans['values'].value_counts(1)

In [ ]:
df_sig_primary_cistrans['-log10(p-wald)'] = [-np.log10(i) for i in df_sig_primary_cistrans['p_wald']]
df_sig_primary_cistrans.sort_values(by='chr',inplace=True)

### Annotate overlapping genes or nearest gene if the SNP does not fall into any gene


In [ ]:
from genelocator import get_genelocator
gl = get_genelocator('GRCh37', gencode_version=31, coding_only=False, auto_fetch=True)

In [ ]:
genes=[]
for index, row in df_sig_primary_cistrans[['chr', 'ps', 'rs']].drop_duplicates().iterrows():
    chr_number = row['chr']
    position = row['ps']
    rs = row['rs']
    gene = gl.at('chr{}'.format(chr_number), position)
    df_gene = pd.DataFrame.from_dict(gene)
    df_gene['rs']=rs
    genes.append(df_gene)
df_nearest_gene = pd.concat(genes)
df_nearest_gene = df_nearest_gene.groupby('rs', as_index=False)['symbol'].agg(list).rename({'symbol':'located_gene'}, axis=1)
df_nearest_gene['lead_located_gene']=df_nearest_gene['located_gene'].apply(lambda x: x[0] if len(x)>0 else None)

In [ ]:
df_final = df_sig_primary_cistrans.merge(df_nearest_gene, on='rs', how='left')

In [ ]:
df_final[['lead_located_gene', 'Gene name']].drop_duplicates()['lead_located_gene'].value_counts()[:10]

In [ ]:
df_final['vep']=df_final['rs'].map(dict_vep)

In [ ]:
df_final = df_final.merge(final_set[['pqtl_id', 'nr.peptides.sig.spec', 'all.same.direction', 'tier']], on='pqtl_id', how='left')

In [ ]:
# Import Tissue specificity annotation from HPA downloaded Feb 10 2023
file_hpa = 'proteinatlas.tsv'
df_hpa = pd.read_csv(os.path.join(annotation_path, file_hpa), sep='\t')
cols_tokeep = ['Gene', 'Uniprot', 'Protein class', 'RNA tissue specificity', 'RNA tissue specific nTPM']
df_hpa = df_hpa[cols_tokeep]
df_hpa = df_hpa.drop_duplicates(subset='Gene', keep='first')
df_hpa = df_hpa.assign(enriched_tissue=df_hpa['RNA tissue specific nTPM'].str.split(':').str[0])

In [ ]:
df_final = df_final.merge(df_hpa, left_on='Gene name', right_on='Gene', how='left')

In [ ]:
df_final.to_csv(os.path.join(Base, 'analysis/results/df_sig_primary_final.csv'), index=False)

#### Read GEMMA pve

In [ ]:
phenotype_to_proteinID = pd.read_csv(os.path.join(Base,'ProteinID.txt'), sep='\t')

In [ ]:
os.chdir(os.path.join(result_path,'log/'))
my_files = glob.glob('*.txt')
pve_values = []
phenotypes = []
for file in my_files:
    phenotype=file.split('.')[0]
    pve=float(pd.read_csv(os.path.join(result_path, 'log/{}'.format(file))).loc[22].str.split('= ')[0][1])
    pve_values.append(pve)
    phenotypes.append(int(phenotype))
df_pve=pd.DataFrame({'Phenotype ID':phenotypes, 'pve':pve_values}).sort_values(by='pve', ascending=False)

In [ ]:
df_pve = df_pve.merge(phenotype_to_proteinID, on='Phenotype ID', how='left')

In [ ]:
df_pve['phenotype']=df_pve['Protein ID']

In [ ]:
df_final = df_final.merge(df_pve[['phenotype', 'pve']], on='phenotype', how='left')
df_pve['with_pqtl']=np.where(df_pve['Protein ID'].isin(df_final['phenotype'].unique()), True, False)

#### Read GCTA-COJO results

- gcta/1.93.3beta2 (https://gcta.freeforums.net/thread/178/conditional-joint-analysis-using-summary)
- check script 
in /home/projects/cpr_man/people/lilniu/projects/target/data/data_genotype/subset/Formatted/plink_qc4/gemma/analysis
script_run_cojo.sh
script_prepare_cojo.sh
prepare-gcta-cojo.ipynb


In [ ]:
# Combine cojo results for all proteins
cojo_path = result_path + 'assoc/cojo/jma/'
os.chdir(cojo_path)
cojo_files = glob.glob('*.jma.cojo')
files = [pd.read_csv(file, sep='\t') for file in cojo_files]
for df, file in zip(files, cojo_files):
    phenotype = int(file.split('.')[0])
    proteinID = IDmapping_phenotype_to_proteinID[phenotype]
    df['phenotype'] = proteinID
cojo_snps = pd.concat(files, ignore_index=True)
cojo_snps['pqtl_id']=cojo_snps['SNP'].astype(str) +'_'+cojo_snps['phenotype'].astype(str)
cojo_snps = cojo_snps[cojo_snps['phenotype'].isin(df_final['phenotype'].unique())]
protein_to_loci_dict = cojo_snps.groupby('phenotype')['SNP'].apply(list).to_dict()
cojo_snps['rsID']=cojo_snps['SNP'].map(IDmapping_snp_to_rs)
protein_to_indeSNP_dict = cojo_snps.groupby('phenotype')['rsID'].apply(list).to_dict()

In [ ]:
cojo_snps_final = cojo_snps[cojo_snps['phenotype'].isin(df_final['phenotype'].unique())]

In [ ]:
cojo_snps_final['phenotype'].nunique()

#### Prepare COJO results

In [ ]:
new_names = {'Chr':'snp_chr', 'SNP':"SNP (chr_pos_ref_alt)", 'bp':'snp_pos', 'refA':'variant_allele', 
            'b':'beta', 'se':'standard error', 'p':'p-wald','n':'estimated effective sample size'}

In [ ]:
df_cojo_formatted = cojo_snps_final.rename(new_names, axis=1)

In [ ]:
with open (os.path.join(vcf_path, 'cojo_pqtl_id.txt'), 'w') as f:
    for line in cojo_snps['SNP'].unique().tolist():
        f.write(line + '\n')

### Estimate proportion of variance explained

#### Load independent SNP data

In [ ]:
vcf_path ='/home/projects/cpr_man/people/lilniu/projects/target/data/data_genotype/subset/Formatted/plink_qc4/'
# Read VCF file
from src.read_vcf import read_vcf
vcf_file = read_vcf(os.path.join(vcf_path, 'target5_cojo.vcf'))
vcf_file.columns = vcf_file.columns.str.split('_').str[0]
vcf_file = vcf_file.set_index('ID')

In [ ]:
df_vcf = vcf_file.iloc[:, 8:]
df_vcf = df_vcf.rename_axis('SNP ID', axis=0)
df_vcf = df_vcf.rename_axis('Participant ID', axis=1).T
df_vcf=df_vcf.replace({'1/1':2, '1/0':1, '0/1':1, '0/0':0})

#### Load proteomics data

In [ ]:
data_prot = pd.read_csv(os.path.join(Base, 'analysis/dataset/data_combined_int.csv')).set_index('Sample ID')
data_prot['Participant ID']='66-' +data_prot['Participant ID']

#### Regress out sample storage related factors

In [ ]:
from src.statistical_testing import perform_linear_regression
proteins = data_prot.columns[:1216]
meta = data_prot.columns[1216:]
stats_lr, residuals = perform_linear_regression(data_prot, proteins, ['time_to_analysis', 'PC1'])
residuals=pd.DataFrame.from_dict(residuals).rename_axis('Sample ID', axis=0)

In [ ]:
cutter = pd.cut(data_prot['age'], [5, 10, 15, 20], right=False)
data_prot['age_range']=cutter

In [ ]:
data_prot[['age', 'age_range']].sort_values(by='age')[500:550]

In [ ]:
#ran INT for seperate age groups
RE_INT = False
if not RE_INT:
    with open(os.path.join(Base, 'analysis/results/age_group_residuals_int.pkl'), 'rb') as handle:
        residuals_int_dict = pickle.load(handle)

else:
    residuals_int_dict = {}
    for age_range in data_prot['age_range'].unique().dropna(): 
        sample_ids = data_prot[data_prot['age_range']==age_range].index
        df_residuals = residuals.loc[sample_ids]
        new_df = []
        for protein in tqdm(proteins):
            new_df.append(pd.DataFrame(rank_INT(df_residuals[protein], stochastic=False), columns=[protein]))
        residuals_int = pd.concat(new_df, axis=1)
        residuals_int.rename_axis('Sample ID', axis=0, inplace=True)
        residuals_int_dict[age_range]=residuals_int

    #ran INT for whole dataset
    new_df = []
    for protein in tqdm(proteins):
        new_df.append(pd.DataFrame(rank_INT(residuals[protein], stochastic=False), columns=[protein]))
    residuals_int = pd.concat(new_df, axis=1)
    residuals_int.rename_axis('Sample ID', axis=0, inplace=True)
    residuals_int_dict['5-20']=residuals_int
    with open(os.path.join(Base, 'analysis/results/age_group_residuals_int.pkl'), 'wb') as handle:
        pickle.dump(residuals_int_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)

#### Perform linear regression

In [ ]:
proteins_w_pqtl = df_final['phenotype'].unique().tolist()

In [ ]:
data_int_age_ranges = [residuals_int_dict[age_range] for age_range in residuals_int_dict.keys()]

In [ ]:
data_join_age_ranges = [df.join(data_prot[meta]).set_index('Participant ID').join(df_vcf, how='inner') for df in data_int_age_ranges]

In [ ]:
RE_ESTIMATE = False
if not RE_ESTIMATE:
    df_pve_all = pd.read_csv(os.path.join(Base, 'analysis/results/pve_estimation_all_age_ranges.csv'))
else:
    pve_age_ranges = []
    age_ranges = ['5-10', '10-15', '15-20', 'full']
    
    for age_range, DATA in zip(age_ranges, data_join_age_ranges):
        results=[]
        
        for protein in tqdm(proteins_w_pqtl):
            snps = protein_to_loci_dict[protein]
            covar_include = snps
            stats_lr, _ = perform_linear_regression(DATA, [protein], snps)
            r2=stats_lr['adj_r2'][0]
            r2s=[r2]
            r2_increments=[r2]
            
            for factor in [['sex'], ['age'], ['overweight'], ['z_BMI.Nysom', 'overweight*z_BMI.Nysom']]:
                stats_lr, _ = perform_linear_regression(DATA, [protein], covar_include + factor)
                r2_new=stats_lr['adj_r2'][0]
                increment = r2_new-r2s[-1]
        
                if increment<=0:
                    r2_increments.append(0)
                    r2s.append(r2s[-1])
                else:
                    r2_increments.append(increment)
                    r2s.append(r2_new)
                    covar_include = covar_include + factor
                    
            models=['snps', 'sex', 'age', 'obesity', 'BMI-SDS&obesity*BMI-SDS']
            result=pd.DataFrame({'phenotype':protein,'r2':r2s, 'r2_increment': r2_increments, 'model':models})
            result['age_range']=age_range
            results.append(result)
        #concatenate results    
        df_perexp=pd.concat(results)
        pve_age_ranges.append(df_perexp)
        
    # Concatenate PVEs for all age ranges
    df_pve_all = pd.concat(pve_age_ranges)
    df_pve_all.to_csv(os.path.join(Base, 'analysis/results/pve_estimation_all_age_ranges.csv'), index=False)

In [ ]:
from src.config import lookup

In [ ]:
df_pve_all = df_pve_all[df_pve_all['phenotype'].isin(df_final['phenotype'].unique())]
df_pve_wide = df_pve_all.set_index(['phenotype', 'age_range', 'model'])[['r2_increment']].unstack(level=-2).reset_index()
df_pve_wide.columns = ['phenotype', 'model', '10-15', '15-20', '5-10', 'full']
df_pve_wide=df_pve_wide[df_pve_wide['model']!='full']

In [ ]:
from mycolorpy import colorlist as mcp
colors = mcp.gen_color(cmap="Accent", n=10)

In [ ]:
#models1=['wo_snps','wo_sex', 'wo_age',  'wo_BMI-SDS','wo_obesity&obesity*BMI-SDS', ]
models1=['snps', 'sex', 'age', 'obesity', 'BMI-SDS&obesity*BMI-SDS']
color=['#53BBD5', 'gold', 'green', 'crimson', 'royalblue', ]

#### Source data Fig. 4

In [ ]:
sourcedata_fig4 = df_pve_wide[df_pve_wide['model']=='snps'][['phenotype', 
                                                             '10-15', '15-20', 
                                                             '5-10', 'full']].reset_index(drop=True)
#sourcedata_fig4.to_excel('tables/SourceData_Figure4.xlsx')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
plt.subplots_adjust(wspace=0.4, hspace=None)
palette='Accent'
for ax, x, y in zip(axes, ['5-10', '5-10', '10-15'], ['10-15', '15-20', '15-20']):
#     sns.scatterplot(x=x, y=y, data=df_pve_wide, hue='model', hue_order=models1, 
#                     ax=ax, palette=dict(zip(models1, color)), legend=False)
    sns.scatterplot(x=x, y=y, data=df_pve_wide[df_pve_wide['model']=='snps'],
                    ax=ax, color='#53BBD5')
    corr = lookup(df_pve_wide, 'model', 'snps')[[x, y]].corr().iloc[1, 0].round(2)
    ax.annotate('Pear. corr. coef. \nfor pQTLs: {}'.format(corr), xy=(0.4, 0.05))
    #ax.set_ylabel('Proportion of variance explained')
    ticks = [0, 0.2, 0.4, 0.6, 0.8]
    labels = [0, 20, 40, 60, 80]
    ax.set_xticks(ticks=ticks, labels=labels)
    ax.set_yticks(ticks=ticks, labels=labels)
    ax.set_xlabel('Age {}\nProportion of variance explained'.format(x))
    ax.set_ylabel('Age {}\nProportion of variance explained'.format(y))
    
for ax in axes:
    sns.lineplot(x=(0, 0.8), y=(0, 0.8), ax=ax, color='gray')
plt.legend(bbox_to_anchor=(1,1))
plt.rcParams['pdf.fonttype'] = 42
fig.savefig(os.path.join(figure_path, 'pve_age_range.pdf'), dpi=120, bbox_inches='tight')

In [ ]:
df_pve_fullset = lookup(df_pve_all, 'age_range', 'full')

In [ ]:
df_pve_fullset_wide = df_pve_fullset.pivot(index='phenotype', columns='model', values='r2_increment')
df_proportion = df_pve_fullset_wide.merge(lookup(df_pve_fullset, 'model', 'BMI-SDS&obesity*BMI-SDS')[['phenotype', 'r2']], 
                                          on='phenotype', how='left')
df_proportion = df_proportion.sort_values(by='snps', ascending=False).set_index('phenotype')

In [ ]:
df_proportion['Gene name']=df_proportion.index.str.split('_').str[1]

In [ ]:
df_proportion['r2_rest']=df_proportion.apply(lambda x:x['r2'] - x['snps'], axis=1)
df_proportion['snps > rest']=df_proportion['snps']>df_proportion['r2_rest']

In [ ]:
df_proportion['snps > rest'].value_counts(1)

In [ ]:
cistranscount = df_final.groupby(['phenotype'])['values'].value_counts().unstack().sort_values(by=['cis'], ascending=False)
trans_only = cistranscount[cistranscount['trans'].notnull() & cistranscount['cis'].isnull()]

In [ ]:
df_proportion['trans_only']=np.where(df_proportion.index.isin(trans_only.index), True, False)

In [ ]:
x_label_color = df_proportion['trans_only'].map({True:'gray', False:'black'})

In [ ]:
from adjustText import adjust_text

In [ ]:
fig, ax=plt.subplots(figsize=(11, 3))
df_proportion[models1].apply(lambda x:x*100).plot(kind='bar', stacked=True, ax=ax, color=color)
plt.ylabel('Proportion of variance explained (%)')
plt.rcParams['pdf.fonttype'] = 42
plt.legend(bbox_to_anchor=(1,1))
texts = []
proteins_to_annotate = ['ANG', 'LBP', 'PZP', 'IGFBP3', 'CFH', 'LGALS3BP', 
                       'ADIPOQ', 'C3', 'INHBC', 'CRP', 'APCS', 'CFI', 'PRG4', 'SHBG', 'IGF1', 'APOA1']
indexes = lookup(df_proportion, 'Gene name', proteins_to_annotate).index
for index in indexes:
    x=df_proportion.index.get_loc(index)
    y=df_proportion.loc[index]['r2']*100
    l=df_proportion.loc[index]['Gene name']
    texts.append(plt.text(x, y, l, size=8, ha='center', va='center'))
#adjust_text(texts, arrowprops=dict(arrowstyle="->", color='r', lw=0.5))
fig.savefig(os.path.join(figure_path, 'pve.pdf'), dpi=120, bbox_inches='tight')

#### Revision2

In [ ]:
# Export proportion as a supplementary table
df_export = df_proportion[models1]
df_export['independent loci'] = df_export.index.map(protein_to_indeSNP_dict)

### Report number of pQTLs identified

In [ ]:
# Compute the number of significant associations, proteins, and SNPs
gw_sig_threshold = 5e-8
study_wide_sig_threshold = gw_sig_threshold / 1216

for threshold_type, threshold in [('genome-wide', gw_sig_threshold), ('study-wide', study_wide_sig_threshold)]:
    significant = df_final[df_final['p_wald'] < threshold]
    nr_associations = significant.shape[0]
    nr_proteins = significant['Protein ID'].nunique()
    nr_snps = significant['rs'].nunique()
    print(f"At {threshold_type} significance threshold,\nafter clump and HLA region removed,\n{nr_associations} primary associations were identified,\ninvolving {nr_proteins} proteins, and \n{nr_snps} SNPs\n")

#### Assess post-clump pair-wise LD between primary pQTLs of the same protein

In [ ]:
# Import LD results
# plink --file target5_primary_out --r2 --ld-window 99999 --ld-window-kb 99999 --ld-window-r2 0
file = '/home/projects/cpr_man/people/lilniu/projects/target/data/data_genotype/subset/Formatted/plink_qc4/plink.ld'
df_ld = pd.read_csv(file, sep='\s+')

nr_snps = len(set(df_ld['SNP_A'].unique().tolist() + df_ld['SNP_B'].unique().tolist()))
print('Number of SNPs pair-wise LD calculation: {}'.format(nr_snps))

In [ ]:
# Group the DataFrame by phenotype and extract the rs column as a Series
grouped = df_final.groupby('phenotype')['rs']

# Create a dictionary comprehension that maps each phenotype to a list of rs values
rs_dict = {phenotype: list(rs_values) for phenotype, rs_values in grouped}

In [ ]:
# Extract pairwise LD results for each protein
ld_primary = []
for phenotype in df_final['phenotype'].unique():
    pqtls = rs_dict[phenotype]
    mask = df_ld['SNP_A'].isin(pqtls) & df_ld['SNP_B'].isin(pqtls)
    df = df_ld.loc[mask].copy()
    df['phenotype'] = phenotype
    ld_primary.append(df)

# Concatenate pairwise results from all proteins 
df_ld_postclump = pd.concat(ld_primary)

In [ ]:
# Extract pairwise LD with r2 <=0.2
le = df_ld_postclump[df_ld_postclump['R2']<=0.2]
le_rate = pd.DataFrame(le.groupby('phenotype')['R2'].count()/df_ld_postclump.groupby('phenotype')['R2'].count())
le_rate.columns=['%_R2<0.2']
le_rate_mean = le_rate['%_R2<0.2'].mean()
print('On average, {}% of pair-wise LD have a R2 <=0.2'.format(round(le_rate_mean, 2)*100))

In [ ]:
fig, ax = plt.subplots(figsize=(4,4))
df_ld_postclump['R2'].hist(bins=50)
plt.xlabel('LD (r2)')

#### Extract missense variants to eliminate artefactual pQTLs

In [ ]:
# Filter and select relevant columns from vep_output
coding_cons = vep_output.loc[vep_output['Consequence'] == 'missense_variant', ['#Uploaded_variation', 'SYMBOL', 'Protein_position', 'Amino_acids']].drop_duplicates()

# Extract SNP to affected protein mapping
IDmapping_snp_to_affectedprot = coding_cons[['#Uploaded_variation', 'SYMBOL']].drop_duplicates().set_index('#Uploaded_variation')['SYMBOL'].to_dict()

# Map SNP IDs to affected proteins in df_sig
df_sig_coding_cons = df_sig.assign(affected_protein=df_sig['rs'].map(IDmapping_snp_to_affectedprot))

# Select only the rows where the affected protein matches the gene name
df_sig_coding_tocheck = df_sig_coding_cons.loc[df_sig_coding_cons['Gene name'] == df_sig_coding_cons['affected_protein'], :].sort_values('phenotype')

# Save file to path
df_sig_coding_tocheck.to_csv(os.path.join(Base, 'analysis/alphamap/df_sig_coding_tocheck.csv'), index=False)

# Extract coding consequences of all missense variants and save file to path
coding_cons_allsig = coding_cons[coding_cons['#Uploaded_variation'].isin(df_sig_coding_tocheck['rs'].unique())]
coding_cons_allsig.to_csv(os.path.join(Base,'analysis/alphamap/coding_cons_allsig.csv'), index=False)

#### Plot figures

##### Figure 3c

In [ ]:
df_ve = lookup(consequences, '#Uploaded_variation', df_final['rs'].unique())

# Compute values for the pie chart
df_pie = df_ve['Consequence'].value_counts() / df_ve['#Uploaded_variation'].nunique()
df_pie['others'] = df_pie[df_pie <= 0.012].sum()
toplot_pie = (df_pie[df_pie > 0.012] * 100).round().astype(int)

# Generate labels for the pie chart
labels = [f"{index}: {value}%" for index, value in toplot_pie.items()]

# Generate colors for the pie chart
from mycolorpy import colorlist as mcp
colors = mcp.gen_color(cmap="Paired", n=toplot_pie.shape[0])

# Plot the pie chart
fig, ax = plt.subplots()
ax.pie(toplot_pie, colors=colors, wedgeprops={"edgecolor":"black", 'linewidth': 1, 'linestyle': 'solid', 'antialiased': True})
ax.legend(labels, loc=(1, 0.1), labelcolor='black')
plt.rcParams['pdf.fonttype'] = 42
fig.savefig(os.path.join(figure_path, 'Fig.3C.pdf'), dpi=120, bbox_inches='tight')

In [ ]:
dfg = pd.DataFrame(toplot_pie).reset_index()
dfg.columns=['VEP annotation','Percentage (%)']

In [ ]:
# Compute the number of proteins per SNP and SNPs per protein
protein_per_snp = df_final.groupby('rs')['Protein ID'].count().value_counts()
snp_per_protein = df_final.groupby('Protein ID')['rs'].count().value_counts()

# Compute the number of chromosomes per protein and identify the proteins with more than one chromosome
chr_per_protein = df_final.groupby('Protein ID')['chr'].nunique()
# Sort proteins by the number of chromosomes
chr_per_protein = chr_per_protein.sort_values(ascending=False).to_frame(name='chr')
proteins_with_multiple_chromosomes = chr_per_protein[chr_per_protein > 1].shape[0]

# Prepare data for plotting
snp_per_protein_plot = snp_per_protein[snp_per_protein.index.isin(range(1, 11))].sort_index()
snp_per_protein_plot['>10'] = snp_per_protein[snp_per_protein.index > 10].sum()

protein_per_snp_plot = protein_per_snp[protein_per_snp.index.isin(range(1, 7))].sort_index()
protein_per_snp_plot['>6'] = protein_per_snp[protein_per_snp.index > 6].sum()

##### Fig. 3e

In [ ]:
cis_trans_prop =df_final['values'].value_counts(1, dropna=False).round(2)*100
cis_trans_prop = cis_trans_prop.rename({'cis':'Cis', 'trans':'Trans'})
cis_trans_nr = df_final['values'].value_counts(dropna=False)
labels=[cis_trans_prop.index[i] +  ": {} ({}%)".format(cis_trans_nr.iloc[i], round(cis_trans_prop.iloc[i])) for i in np.arange(cis_trans_prop.shape[0])]

fig, ax = plt.subplots(figsize=(2,2))
colors=['white', '#53BBD5', 'gray']
plt.pie(cis_trans_prop, colors=colors,
        wedgeprops={"edgecolor":"black",'linewidth': 1, 'linestyle': 'solid', 'antialiased': True});
plt.legend(labels, loc=(-0.2,-0.5), labelcolor='black', fontsize=12)
plt.rcParams['pdf.fonttype'] = 42
plt.savefig(os.path.join(figure_path, 'Fig.3E.pdf'), dpi=120, bbox_inches='tight')

#### how many proteins have cis, trans only and cis/trans associations

In [ ]:
cistranscount = df_final.groupby(['phenotype'])['values'].value_counts().unstack().sort_values(by=['cis'], ascending=False)

In [ ]:
cis_only = cistranscount[cistranscount['cis'].notnull() & cistranscount['trans'].isnull()]
trans_only = cistranscount[cistranscount['trans'].notnull() & cistranscount['cis'].isnull()]
both = cistranscount[(cistranscount['cis'].notnull())&(cistranscount['trans'].notnull())]
ctcount = pd.Series({'cis_only': len(cis_only), 'trans_only': len(trans_only), 'cis/trans': len(both)})

In [ ]:
df_plot = df_final.copy()
df_plot['ind']=range(df_plot.shape[0])
df_plot.chr = df_plot.chr.astype('category')
df_plot_grouped = df_plot.groupby('chr')

##### Fig. 3f-h

In [ ]:
#### Source data
dfd = pd.DataFrame(snp_per_protein_plot).reset_index()
dfd.columns=['Nr. associated SNPs per protein', 'Nr. pQTLs']
dfe = pd.DataFrame(protein_per_snp_plot).reset_index()
dfe.columns=['Nr. associated proteins per SNP', 'Nr. pQTLs']
dff = pd.DataFrame(ctcount).reset_index()
dff.columns = ['Category', 'Nr. proteins']

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(8,2))
fig.subplots_adjust(wspace=0.4)

# Plot number of associated proteins per SNP
axs[0].bar(x=np.arange(7), height=protein_per_snp_plot, 
        facecolor='#53BBD5', edgecolor='#3B5488', width=0.6)
axs[0].set_xticks(ticks=np.arange(7), labels=[i for i in np.arange(1, 7)] + ['>6'], rotation=45);
axs[0].set_xlabel('Number of associated\nproteins per SNP', fontsize=12)
axs[0].set_ylabel('Number of pQTLs', fontsize=12)

axs[1].bar(x=np.arange(11), height=snp_per_protein_plot, 
          facecolor='#53BBD5', edgecolor='#3B5488', width=0.6)

# Plot number of associated SNPs per protein
axs[1].set_xticks(ticks=np.arange(11), labels=[i for i in np.arange(1, 11)] + ['>10'], rotation=45);
axs[1].set_xlabel('Number of associated\n SNPs per protein', fontsize=12)
axs[1].set_ylabel('Number of pQTLs', fontsize=12)


axs[2].bar(x=ctcount.index, height=ctcount, width=0.4, facecolor='#53BBD5', edgecolor='#3B5488')
axs[2].set_ylabel('Number of proteins', fontsize=12)
for bar in axs[2].patches:
    plt.annotate(bar.get_height(), xy=(bar.get_x() + bar.get_width()/2, bar.get_height()), 
                ha='center', va='center', xytext=(0, -9), textcoords='offset points',
                 color='white', fontsize=12)
axs[2].set_xticks(ticks=[0,1,2], labels=['Cis only', 'Trans only', 'Cis/Trans']);

plt.rcParams['pdf.fonttype'] = 42
plt.savefig(os.path.join(figure_path, 'Fig.3GH.pdf'), dpi=120, bbox_inches='tight')

##### Figure 3b

In [ ]:
df_toplot = df_final.copy().dropna(subset=['Transcription start site (TSS)'])

In [ ]:
# Read chromosome length
df_chr_len = pd.read_csv(os.path.join(annotation_path, 'GRCh37_genome_length.txt'), sep='\t', header=None, names=['chr', 'total_length_bp', 'Genbank_accession', 'Refseq_accession'])
# Remove sex chromosomes
df_chr_len['chr']=df_chr_len['chr'].replace({'X':23, 'Y':24})
df_chr_len=df_chr_len[df_chr_len['chr']!=24]
df_chr_len = df_chr_len.astype({'chr': int})


cum_chr_length = np.cumsum(df_chr_len.set_index('chr')['total_length_bp'])
cum_chr_length.loc[0] = 0

xticks = [(cum_chr_length[i] - cum_chr_length[i - 1]) / 2 + cum_chr_length[i - 1] for i in range(1, 23)]

major_ticks = [xticks[i] for i in [4, 9, 14, 19]]


In [ ]:
new_pos_snp = [cum_chr_length[i-1] for i in df_toplot['chr']] + df_toplot['ps'] 
new_pos_prot = [cum_chr_length[int(i)-1] for i in df_toplot['chr_protein']] + df_toplot['Transcription start site (TSS)'] 
df_toplot = df_toplot.assign(ps_snp = new_pos_snp, ps_prot = new_pos_prot )

In [ ]:
import plotly.express as px

In [ ]:
df_final.groupby('lead_located_gene')['phenotype'].nunique().sort_values(ascending=False).head(20)

In [ ]:
px.scatter(data_frame=df_toplot, x='ps_snp', y='ps_prot', color='values', hover_name = 'pqtl_id')

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))

# Create scatter plots for red dots (cis)
sns.scatterplot(x='ps_snp', y='ps_prot', data=df_toplot[df_toplot['values'] == 'cis'],
                color='red', s=20, legend=None, ax=ax)

# Create scatter plots for blue dots (trans) on top of the red dots
sns.scatterplot(x='ps_snp', y='ps_prot', data=df_toplot[df_toplot['values'] == 'trans'],
                color='#3B5488', s=20, legend=None, ax=ax)

plt.xticks(sorted(cum_chr_length), labels = np.arange(1, 25))
plt.yticks(sorted(cum_chr_length), labels = np.arange(1, 25))

plt.xlabel('pQTL position', fontsize=14);
plt.ylabel('Protein coding gene position', fontsize=14);
plt.xlim(0, cum_chr_length.max()+1e8)
plt.ylim(0, cum_chr_length.max()+1e8)
plt.grid(ls ='--');

plt.rcParams['pdf.fonttype'] = 42
plt.savefig(os.path.join(figure_path, 'cis_vs_trans.pdf'), dpi=120, bbox_inches='tight')

### Protein characterization

### What factors affect the identification of genetic associations for a protein

In [ ]:
RE_SEARCH = False
if not RE_SEARCH:
    df_mw = pd.read_pickle(os.path.join(Base, 'analysis/annotation/molecular_weight.pkl'))
else:
    dict_mw = {}
    for protein in tqdm(df_sig['Protein ID'].unique()):
        data = urlopen("http://www.uniprot.org/uniprot/" + protein + ".txt").read().decode()
        result = data.split('SQ   ')[-1] #here all the info you need
        mw = int(result.split(';')[1].strip().split()[0]) #extract the molecular weigth
        dict_mw[protein]=mw

    df_mw = pd.DataFrame.from_dict(dict_mw, orient='index').reset_index()
    df_mw.columns=['Protein ID', 'MW']
    df_mw.to_pickle(os.path.join(Base, 'analysis/annotation/molecular_weight.pkl'))

In [ ]:
df_cv=pd.read_csv(os.path.join(Base, 'analysis/annotation/df_cv.csv'))
pep_avail=pd.read_csv(os.path.join(Base,'analysis/annotation/pep_avail.csv'))
pep_per_protein=pd.read_csv(os.path.join(Base, 'analysis/annotation/nr_peptide_per_protein.csv'))

In [ ]:
pep_per_protein['Protein ID']=pep_per_protein['Protein IDs'].str.split(';').str[0]

In [ ]:
df_cv['pqtl']=np.where(df_cv['ProteinID_Genename'].isin(df_final['phenotype'].unique()), True, False)
df_cv['Protein ID']=df_cv['ProteinID_Genename'].str.split('_').str[0]
df_cv=df_cv.merge(pep_avail[['Protein ID', 'nr.peptides.avail']], on='Protein ID', how='left')

In [ ]:
df_cv=df_cv.merge(pep_per_protein[['Protein ID', 'PEP.StrippedSequence']], on='Protein ID', how='left')

In [ ]:
df_cv=df_cv.merge(df_mw, on='Protein ID', how='left')

In [ ]:
df_cv=df_cv.sort_values(by='Coefficient of variation', ascending=True)
cum_nr = df_cv['pqtl'].astype(int).cumsum()
nr_rows = range(1, len(df_cv) + 1)
df_cv['cumulative.%.pqtl.cv'] = cum_nr / nr_rows

In [ ]:
df_cv=df_cv.sort_values(by='nr.peptides.avail', ascending=False)
cum_nr = df_cv['pqtl'].astype(int).cumsum()
nr_rows = range(1, len(df_cv) + 1)
df_cv['cumulative.%.pqtl.nr.pep'] = cum_nr / nr_rows

In [ ]:
df_cv=df_cv.sort_values(by='Protein abundance [Log10]', ascending=False)
cum_nr = df_cv['pqtl'].astype(int).cumsum()
nr_rows = range(1, len(df_cv) + 1)
df_cv['cumulative.%.pqtl.abundance'] = cum_nr / nr_rows

In [ ]:
df_cv=df_cv.merge(df_hpa, how='left', left_on='Gene name', right_on='Gene')

In [ ]:
cv_bins = pd.cut(df_cv['Coefficient of variation'], 
                 bins=[0, 0.2, 0.4, 0.6, df_cv['Coefficient of variation'].max()])
abundance_bins = pd.cut(df_cv['Protein abundance [Log10]'], 
                        bins=[0, 2, 3,4, 5, df_cv['Protein abundance [Log10]'].max()])

In [ ]:
nr_pep_bins = pd.cut(df_cv['PEP.StrippedSequence'],
                    bins=[0, 5, 10, 20, 30, df_cv['PEP.StrippedSequence'].max()])

In [ ]:
nr_pep_bins.value_counts()

In [ ]:
df_cv['cv_bins']=cv_bins.astype(str)
df_cv['abundance_bins']=abundance_bins.astype(str)
df_cv['nr_pep_bins']=nr_pep_bins.astype(str)
dfa = pd.DataFrame(df_cv.groupby('cv_bins')['pqtl'].value_counts(1))*100
dfa.columns=['proportion_w_pqtl']
dfa=dfa.reset_index()

dfb=pd.DataFrame(df_cv.groupby('abundance_bins')['pqtl'].value_counts(1))*100
dfb.columns=['proportion_w_pqtl']
dfb=dfb.reset_index()

dfc=pd.DataFrame(df_cv.groupby('nr_pep_bins')['pqtl'].value_counts(1))*100
dfc.columns=['proportion_w_pqtl']
dfc=dfc.reset_index()

dfa=dfa[dfa['pqtl']]
dfb=dfb[dfb['pqtl']]
dfc=dfc[dfc['pqtl']]
dfb=dfb.sort_values(by='proportion_w_pqtl', ascending=False)

In [ ]:
order=['(0, 5]', '(5, 10]', '(10, 20]', '(20, 30]', '(30, 297]']
dfc['order']=dfc['nr_pep_bins'].map(dict(zip(order, np.arange(len(order)))))

In [ ]:
dfc=dfc.sort_values(by='order', ascending=False)

##### Fig. 3i-k

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(8, 2))
fig.subplots_adjust(wspace=0.5)
order=dfb['abundance_bins'][::-1]
for ax, df, col in zip(axes, [dfa, dfb, dfc], ['cv_bins', 'abundance_bins', 'nr_pep_bins']):         
    ax.bar(x=col, height='proportion_w_pqtl', data=df, facecolor='#53BBD5', edgecolor='#3B5488', width=0.6)
    ax.set_ylabel('Proportion of proteins\n with pQTL (%)')
    xlabels = ax.get_xticklabels()
    ax.set_xticklabels(xlabels, rotation=40,)
plt.rcParams['pdf.fonttype'] = 42
plt.savefig(os.path.join(figure_path, 'deep_dive_mm.pdf'), dpi=120, bbox_inches='tight')

#### Source data for Fig. 3 

In [ ]:
# Write table 1-3, and 6 to supplementary tables
with pd.ExcelWriter(os.path.join(Base, 'analysis/tables/SourceData_Figure3.xlsx')) as writer:
    df_sig.to_excel(writer, sheet_name='Fig. 3a', index=False)
    dfg.to_excel(writer, sheet_name='Fig. 3c', index=False)
    dfd.to_excel(writer,sheet_name='Fig. 3f', index=False)
    dfe.to_excel(writer,sheet_name='Fig. 3g', index=False)
    dff.to_excel(writer,sheet_name='Fig. 3h', index=False)
    dfa.to_excel(writer,sheet_name='Fig. 3i', index=False)
    dfb.to_excel(writer,sheet_name='Fig. 3j', index=False)
    dfc.to_excel(writer,sheet_name='Fig. 3k', index=False)

In [ ]:
#sns.scatterplot(x='nr.peptides.avail', y='cumulative.%.pqtl.nr.pep', data=df_cv)
fig, axs=plt.subplots(1, 3, figsize=(8,2.))
fig.subplots_adjust(wspace=0.6)
palette={'CV<30%':'#53BBD5', 'rest':'white'}

sns.scatterplot(x='Coefficient of variation', y='cumulative.%.pqtl.cv', 
                data=df_cv, ax=axs[0], color='#53BBD5', edgecolor='gray')

sns.scatterplot(x='Protein abundance [Log10]', 
                y='cumulative.%.pqtl.abundance', 
                data=df_cv,hue='color',
                palette=palette, edgecolor='gray',             
                ax=axs[1])
    
sns.scatterplot(x='nr.peptides.avail', y='cumulative.%.pqtl.nr.pep', 
                data=df_cv, ax=axs[2], hue='color', edgecolor='gray', palette=palette)
#axs[0].set_xlim(0, 2)
for ax in axs:
    ax.set_ylim(-0.1, 1.1)
    ax.set_ylabel('Cumulative proportion of\n proteins with pQTLs')
axs[1].invert_xaxis() 
axs[2].invert_xaxis() 

plt.rcParams['pdf.fonttype'] = 42
plt.savefig(os.path.join(figure_path, 'deep_dive.pdf'), dpi=120, bbox_inches='tight')

In [ ]:
import scipy
cols_tokeep = ['nr.peptides.avail', 'Coefficient of variation', 'MW']
for i in cols_tokeep:
    print(scipy.stats.ttest_ind(a=lookup(df_cv,'pqtl', True)[i].dropna(), 
                     b=lookup(df_cv, 'pqtl', False)[i].dropna()))

In [ ]:
palette = {True: "steelblue", False: "lightgray"}

In [ ]:
plt.rcParams.update({'font.size': 12})
fig, axs = plt.subplots(1, 3, figsize=(15, 3))
sns.violinplot(x='pqtl', y='nr.peptides.avail', data=df_cv, cut=0, ax=axs[0], palette=palette).set(yscale='log')
sns.histplot(data=df_cv, x='Coefficient of variation', hue='pqtl', ax=axs[1],bins=100, palette=palette,kde=True, alpha=0.7)#.set(yscale='log')
#sns.histplot(data=df_cv, x='Coefficient of variation', hue='pqtl', ax=axs[1],
#             palette=palette, element='step', alpha=0.2, linewidth=2)

sns.violinplot(x='pqtl', y='MW', data=df_cv, cut=0, ax=axs[2], palette=palette).set(yscale='log')
axs[1].set_ylabel('nr.proteins')
axs[2].set_ylabel('molecular.weight')
plt.savefig(os.path.join(Base, 'analysis/figures/1.png'), dpi=120, bbox_inches='tight')

# !!!!!!

### Replication in TARGET 1K study

In [ ]:
# Import SNPs in the 1k cohort
onek_vcf_path = '/home/projects/cpr_man/people/lilniu/projects/target/data/data_genotype/subset_1k/Formatted/plink_qc/'
path_1k = onek_vcf_path + 'gemma/analysis/'
bim_1k = onek_vcf_path + 'gemma/target1k5.bim'
snps_1k = pd.read_csv(bim_1k, header=None, sep='\t', names=['chr', 'rs', 'what', 'pos', 'allele1', 'allele0'])

In [ ]:
# Import proteins in the 1k cohort
proteins_1k = pd.read_csv(os.path.join(onek_vcf_path, 'gemma/ProteinID.txt'), sep='\t')
proteins_1k=proteins_1k.rename({'Protein ID':'ProteinID_Genename'}, axis=1)
proteins_1k['Protein ID']=proteins_1k['ProteinID_Genename'].str.split('_').str[0]

In [ ]:
# Check how many protein-SNP pairs can be tested in the 1k cohort
cond1 = df_final['Protein ID'].isin(proteins_1k['Protein ID'].tolist())
cond2 = df_final['rs'].isin(snps_1k['rs'])

In [ ]:
df_to_rep = df_final[cond1 & cond2]
df_to_rep = df_to_rep.assign(pqtl_id=df_to_rep['rs'].astype(str) +'_' + df_to_rep['Protein ID'].astype(str), chr=df_to_rep['chr'].astype(str))
df_to_rep.to_csv(os.path.join(path_1k, 'target_to_replicate.csv'), index=False)

print('{} variant-protein pairs need to be tested'.format(df_to_rep.shape[0]))

In [ ]:
df_to_rep['phenotype'].nunique()

In [ ]:
df_to_rep_repID = pd.DataFrame(df_to_rep['phenotype'].unique(), columns = ['Protein ID'])
df_to_rep_repID.to_csv(os.path.join(path_1k,'proteins_to_replicate.txt'), index=False)

In [ ]:
prot_chr_to_rep = pd.DataFrame(df_to_rep.groupby('Protein ID')['chr'].unique())
prot_chr_to_rep_dict = dict(zip(prot_chr_to_rep.index, prot_chr_to_rep['chr']))

import pickle
with open(os.path.join(path_1k, 'prot_chr_to_rep_1k.pkl'), 'wb') as handle:
    pickle.dump(prot_chr_to_rep_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
# Significant pQTLs in the replication cohort(p<0.05) and assign new columns
# Run subset_1k/Formatted/plink_qc/gemma/analysis/pqtl_1k.ipynb
df_sig_1k = pd.read_csv(os.path.join(crossref_path, 'onek/pqtl_1k_sig_nominal.csv'))
df_sig_1k['Protein ID'] = df_sig_1k['phenotype'].str.split('_').str[0]
df_sig_1k['pqtl_id'] = df_sig_1k['rs'].astype(str) + '_' + df_sig_1k['Protein ID'].astype(str)
df_sig_1k['chr'] = df_sig_1k['chr'].astype(str)
df_sig_1k['maf'] = np.where(df_sig_1k['af'] < 0.5, df_sig_1k['af'].round(3), 0.5)
df_sig_1k['-log10(p-wald)'] = (-np.log10(df_sig_1k['p_wald'])).round(3)
df_sig_1k[['se_3digit', 'beta_3digit']] = df_sig_1k[['se', 'beta']].round(3)

In [ ]:
df_sig_1k = df_sig_1k[df_sig_1k['phenotype'].isin(df_to_rep_repID['Protein ID'])]

In [ ]:
df_to_rep.shape[0]

In [ ]:
df_sig_1k_fdr = df_sig_1k[df_sig_1k['p_wald']<0.05/df_to_rep.shape[0]]

In [ ]:
len(set(df_to_rep['pqtl_id']) & set(df_sig_1k['pqtl_id']))

### Replication
- criteria: a pQTL is considered replicated if the SNP or its proxy in LD (+/1Mb, r2 > 0.2) is also sig. associated with the corresponding protein (p < 0.05)

#### Nominal significance

In [ ]:
#%run /home/projects/cpr_man/people/lilniu/projects/target/data/data_genotype/subset/Formatted/plink_qc4/gemma/analysis/1k_ld.ipynb
# projects/target/data/data_genotype/subset/Formatted/plink_qc4/gemma/analysis/1k_ld.ipynb
with open(os.path.join(onek_vcf_path,'1k_snp_ld.pkl'), 'rb') as handle:
    snp_ld_dict_1k = pickle.load(handle)

In [ ]:
from src.config import annotate_replication
RE_MATCH = False
if not RE_MATCH:
    df_rep_1k = pd.read_csv(os.path.join(crossref_path, 'onek/replicated_in_1k.csv'), low_memory=False)
else:
    df_rep_1k = annotate_replication(df_to_rep, df_sig_1k, snp_ld_dict=snp_ld_dict_1k)
    df_rep_1k.to_csv(os.path.join(crossref_path, 'onek/replicated_in_1k.csv'), index=False)

In [ ]:
nr_replicated = df_rep_1k[df_rep_1k['replicated'].isin(['vicinity', 'exact'])].shape[0]
per_replicated = nr_replicated/df_rep_1k.shape[0]*100
print('{:.0f}% of testable primary pQTLs in the TARGET cohort was replicated in the 1k study.'.format(per_replicated))

In [ ]:
df_rep_1k['replicated'].value_counts(0)

### Adjusting for multiple testing

In [ ]:
RE_MATCH = False
if not RE_MATCH:
    df_rep_1k_fdr = pd.read_csv(os.path.join(crossref_path, 'onek/replicated_in_1k_fdr.csv'), low_memory=False)
else:
    df_rep_1k_fdr = annotate_replication(df_to_rep, df_sig_1k[df_sig_1k['p_wald']<len(df_to_rep)], snp_ld_dict=snp_ld_dict_1k)
    df_rep_1k_fdr.to_csv(os.path.join(crossref_path, 'onek/replicated_in_1k_fdr.csv'), index=False)

In [ ]:
df_rep_1k_fdr['replicated'].value_counts(1)

#### Check correlation in beta statistics between discovery and replication cohort

In [ ]:
df_exact = df_rep_1k[df_rep_1k['replicated']=='exact'][['pqtl_id', 'beta', 'rs', 'phenotype']].merge(df_sig_1k[['pqtl_id', 'beta']], 
                                                                        on='pqtl_id', how='left')
df_exact=df_exact.rename({'beta_x':'beta_2k', 'beta_y':'beta_1k'}, axis=1)
df_exact['beta_2k']=np.float64(df_exact['beta_2k'])
df_exact = df_exact.sort_values(by='beta_2k')
df_exact['pair_id']=np.arange(df_exact.shape[0])

In [ ]:
df_exact_long = df_exact.melt(id_vars=['pqtl_id', 'pair_id'], value_name='beta', var_name='cohort')
df_exact_long['cohort']=df_exact_long['cohort'].map({'beta_2k':'Holbæk_discovery', 'beta_1k':'Holbaek_replication'})
df_exact[['beta_2k', 'beta_1k']].corr()

In [ ]:
corr = round(df_exact[['beta_2k', 'beta_1k']].corr().iloc[0, 1], 2)

In [ ]:
fig, ax = plt.subplots(figsize=(4,4))
b = sns.scatterplot(x='beta_2k', y='beta_1k', data=df_exact, edgecolor='green', color='white')
plt.xlabel('BETA (Holbæk discovery)', fontsize=12)
plt.ylabel('BETA (Holbæk replication)', fontsize=12)
b.tick_params(labelsize=12)
plt.annotate('Pearson r: {}'.format(corr), xy=(0.5, 0.1), xycoords='axes fraction')
b.set_xticks([-1.5, -1.0, -0.5, 0, 0.5, 1, 1.5])
plt.rcParams['pdf.fonttype'] = 42
plt.savefig(os.path.join(figure_path, 'beta_concordance_nominal_1k.pdf'), dpi=120, bbox_inches='tight')

#### Export pQTLs for plotting dose-responses

In [ ]:
with open (os.path.join(onek_vcf_path, 'exact_match_pqtl_id.txt'), 'w') as f:
    for line in df_exact['rs'].tolist():
        f.write(line + '\n')
df_exact.to_csv(os.path.join(onek_vcf_path, 'exact_match_pqtl.csv'), index=False)

In [ ]:
lookup(df_rep_1k, 'Gene name', 'HRG')[['rsID', 'replicated']]

# !!!!!!!

### Replication in ALD study

In [ ]:
# Import SNPs in the ALD study
ald_vcf_path = '/home/projects/cpr_man/people/lilniu/projects/proteogenomics/data/data_genotype/pqtl/data_genotype_subset/plink_qc2/'
path_ald = ald_vcf_path + 'gemma/analysis/'
bim_ald = ald_vcf_path + 'gemma/ald5.bim'
snps_ald = pd.read_csv(bim_ald, header=None, sep='\t', names=['chr', 'rs', 'what', 'pos', 'allele1', 'allele0'])
snps_ald['rs'] = snps_ald['rs'].str.replace(':', "_")

# Import proteins in the ALD study
proteins_ald = pd.read_csv(os.path.join(ald_vcf_path, 'gemma/ProteinID.txt'), sep='\t')
proteins_ald['Protein ID']=proteins_ald['Protein ID'].str.split('_').str[0]

# Check how many protein-SNP pairs can be tested in the 1k cohort
cond1 = df_final['Protein ID'].isin(proteins_ald['Protein ID'].tolist())
cond2 = df_final['rs'].isin(snps_ald['rs'])

df_to_rep = df_final[cond1 & cond2]
df_to_rep = df_to_rep.assign(pqtl_id=df_to_rep['rs'] + df_to_rep['Protein ID'], chr=df_to_rep['chr'].astype(str))
df_to_rep.to_csv(os.path.join(path_ald, 'target_to_replicate.csv'), index=False)
print('{} variant-protein pairs need to be tested'.format(df_to_rep.shape[0]))

In [ ]:
df_to_rep_repID = pd.DataFrame(df_to_rep['phenotype'].unique(), columns = ['Protein ID'])
df_to_rep_repID.to_csv(os.path.join(path_ald,'proteins_to_replicate.txt'), index=False)

In [ ]:
prot_chr_to_rep = pd.DataFrame(df_to_rep.groupby('Protein ID')['chr'].unique())
prot_chr_to_rep_dict = dict(zip(prot_chr_to_rep.index, prot_chr_to_rep['chr']))

import pickle
with open(os.path.join(path_ald, 'prot_chr_to_rep.pkl'), 'wb') as handle:
    pickle.dump(prot_chr_to_rep_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
# Significant pQTLs in the replication cohort(p<0.05) and assign new columns
# significant pQTLs after Bonferroni correction
#df_sig_ald = pd.read_csv(os.path.join(crossref_path, 'ald/pqtl_ald_sig.csv'))
df_sig_ald = pd.read_csv(os.path.join(crossref_path, 'ald/pqtl_ald_sig_nominal.csv'))
df_sig_ald['Protein ID'] = df_sig_ald['phenotype'].str.split('_').str[0]
df_sig_ald['pqtl_id'] = df_sig_ald['rs'] + df_sig_ald['Protein ID']
df_sig_ald['chr'] = df_sig_ald['chr'].astype(str)
df_sig_ald['maf'] = np.where(df_sig_ald['af'] < 0.5, df_sig_ald['af'].round(3), 0.5)
df_sig_ald['-log10(p-wald)'] = (-np.log10(df_sig_ald['p_wald'])).round(3)
df_sig_ald[['se_3digit', 'beta_3digit']] = df_sig_ald[['se', 'beta']].round(3)

#### Replication
- criteria: a pQTL is considered replicated if the SNP or its proxy in LD (+/1Mb, r2 > 0.2) is also sig. associated with the corresponding protein (p < 0.05)

In [ ]:
from src.config import annotate_replication
#compute snp_ld_dict with analysis/ald_ld.ipynb
with open(os.path.join(ald_vcf_path,'ald_snp_ld.pkl'), 'rb') as handle:
    snp_ld_dict = pickle.load(handle)

In [ ]:
len(snp_ld_dict)

In [ ]:
RE_MATCH = False
if not RE_MATCH:
    df_rep_ald = pd.read_csv(os.path.join(crossref_path, 'ald/replicated_in_ald.csv'), sep='\t', low_memory=False)
else:
    df_rep_ald = annotate_replication(df_to_rep, df_sig_ald, snp_ld_dict=snp_ld_dict)
    df_rep_ald.to_csv(os.path.join(crossref_path, 'ald/replicated_in_ald.csv'), sep='\t', index=False)

In [ ]:
nr_replicated = df_rep_ald[df_rep_ald['replicated'].isin(['vicinity', 'exact'])].shape[0]
per_replicated = nr_replicated/df_rep_ald.shape[0]*100
print('{:.0f}% of testable primary pQTLs in the TARGET cohort was replicated in the ALD study.'.format(per_replicated))

#### Adjusting for multiple testing

In [ ]:
RE_MATCH = False
if not RE_MATCH:
    df_rep_ald_fdr = pd.read_csv(os.path.join(crossref_path, 'ald/replicated_in_ald_fdr.csv'), low_memory=False)
else:
    df_rep_ald_fdr = annotate_replication(df_to_rep, df_sig_ald[df_sig_ald['p_wald']<0.05/len(df_to_rep)], snp_ld_dict=snp_ld_dict)
    df_rep_ald_fdr.to_csv(os.path.join(crossref_path, 'ald/replicated_in_ald_fdr.csv'), index=False)

#### Check correlation in beta statistics between discovery and replication cohort

In [ ]:
df_exact = df_rep_ald[df_rep_ald['replicated']=='exact'][['pqtl_id', 'beta', 'rs', 'phenotype']].merge(df_sig_ald[['pqtl_id', 'beta']], 
                                                                        on='pqtl_id', how='left')
df_exact=df_exact.rename({'beta_x':'beta_target', 'beta_y':'beta_ald'}, axis=1)
df_exact['beta_target']=np.float64(df_exact['beta_target'])
df_exact = df_exact.sort_values(by='beta_target')
df_exact['pair_id']=np.arange(df_exact.shape[0])

In [ ]:
df_rep_ald['replicated'].value_counts()

In [ ]:
df_exact_long = df_exact.melt(id_vars=['pqtl_id', 'pair_id'], value_name='beta', var_name='cohort')
df_exact_long['cohort']=df_exact_long['cohort'].map({'beta_target':'Holbæk', 'beta_ald':'GALA-ALD'})
df_exact[['beta_target', 'beta_ald']].corr()

In [ ]:
corr = round(df_exact[['beta_target', 'beta_ald']].corr().iloc[0, 1], 2)

In [ ]:
fig, ax = plt.subplots(figsize=(4,4))
b = sns.scatterplot(x='beta_target', y='beta_ald', data=df_exact, edgecolor='goldenrod', color='white')
plt.xlim(-1.7, 1.7)
plt.ylim(-1.7, 1.7);
plt.xlabel('BETA (Holbæk)', fontsize=12)
plt.ylabel('BETA (GALA-ALD)', fontsize=12)
b.tick_params(labelsize=12)
plt.annotate('Pearson r: {}'.format(corr), xy=(0.5, 0.1), xycoords='axes fraction')
b.set_xticks([-1.5, -1.0, -0.5, 0, 0.5, 1, 1.5])
plt.rcParams['pdf.fonttype'] = 42
plt.savefig(os.path.join(figure_path, 'beta_concordance_nominal.pdf'), dpi=120, bbox_inches='tight')

#### Replicated in 1k but not in ALD

In [ ]:
df_rep_1k['replicated_binary']=np.where(df_rep_1k['replicated']=='no', 0, 1)
df_rep_1k['pqtl_id2']=df_rep_1k['pqtl_id']+'_'+df_rep_1k['Gene name']
df_rep_ald['replicated_binary']=np.where(df_rep_ald['replicated']=='no', 0, 1)
df_rep_ald['pqtl_id2']=df_rep_ald['rs']+'_'+df_rep_ald['phenotype']

In [ ]:
dfa=df_rep_1k[['replicated_binary', 'beta_abs']]
dfa['cohort']='1k'
dfb=df_rep_ald[['replicated_binary', 'beta_abs']]
dfb['cohort']='ald'

In [ ]:
dfc=pd.concat([dfa, dfb])

In [ ]:
dfc['beta_abs']=dfc['beta_abs'].astype(float)

In [ ]:
df_rep_1k['tier']=df_rep_1k['pqtl_id2'].map(dict(zip(final_set['pqtl_id'], final_set['tier'])))
df_rep_ald['tier']=df_rep_ald['pqtl_id2'].map(dict(zip(final_set['pqtl_id'], final_set['tier'])))

In [ ]:
fig, ax=plt.subplots(figsize=(5, 4))
sns.violinplot(x='cohort', hue='replicated_binary', y='beta_abs', 
               data=dfc, cut=0, palette={1:'green', 0:'gray'})
plt.rcParams['pdf.fonttype'] = 42
plt.savefig(os.path.join(figure_path, 'not_replicated.pdf'), dpi=120, bbox_inches='tight')

#### Prepare Supplementary Table 13 and 14

In [ ]:
# Replication results in 1k
df = df_rep_1k.sort_values(by='replicated', ignore_index=True)
df[df['replicated']=='exact'].to_csv(os.path.join(Base, 'analysis/dataset/dataset_replicated.csv'), index=False)
df['maf']=[i if i<=0.5 else 1-i for i in df['af']]
df['-log10(p-wald)']=[-np.log10(i) for i in df['p_wald']]

cols_tokeep =['rs', 'chr', 'REF', 'ALT', 'Protein ID','Gene name', 'maf', 'beta', 'se', 'p_wald', 
             'replicated', 'evidence', 'maf_replication cohort', 'beta_replication cohort', 
             'se_replication cohort', 'p_wald_replication cohort', '-log10(p-wald)_replication cohort']
df=df[cols_tokeep]
df_table13 = df.copy()

# Replication results in ALD
df = df_rep_ald.sort_values(by='replicated', ignore_index=True)
df[df['replicated']=='exact'].to_csv(os.path.join(Base, 'analysis/dataset/dataset_replicated.csv'), index=False)
df['maf']=[i if i<=0.5 else 1-i for i in df['af']]
df['-log10(p-wald)']=[-np.log10(i) for i in df['p_wald']]

cols_tokeep =['rs', 'chr', 'REF', 'ALT', 'Protein ID','Gene name', 'maf', 'beta', 'se', 'p_wald', 
             'replicated', 'evidence', 'maf_replication cohort', 'beta_replication cohort', 
             'se_replication cohort', 'p_wald_replication cohort', '-log10(p-wald)_replication cohort']
df=df[cols_tokeep]
df_table14 = df.copy()

#### Export pQTLs for plotting dose-responses

In [ ]:
with open (os.path.join(ald_vcf_path, 'exact_match_pqtl_id.txt'), 'w') as f:
    for line in df_exact['rs'].tolist():
        f.write(line.replace('_', ':') + '\n')
df_exact.to_csv(os.path.join(ald_vcf_path, 'exact_match_pqtl.csv'), index=False)

### Cross reference other studies

In [ ]:
# Import prior studies
prior_studies=pd.read_csv(os.path.join(crossref_path, 'all_studies_tidy.txt'), low_memory=False)
prior_studies.dropna(subset=['GeneSymbol', 'Chr.SNP.hg19', 'Pos.SNP.hg19'], inplace=True)
prior_studies.reset_index(drop=True, inplace=True)
prior_studies=prior_studies.astype({'Pos.SNP.hg19':int,
                                   'Chr.SNP.hg19':str})

# Save results
prior_studies.to_csv(os.path.join(crossref_path, 'prior_studies_pqtl_list.txt'), index=False)

In [ ]:
prior_studies=pd.read_csv(os.path.join(crossref_path, 'all_studies_tidy.txt'), low_memory=False)

In [ ]:
from src.config import estimate_novelty, diff
df_sig4 = estimate_novelty(df_final, prior_studies)

In [ ]:
# Select novel significant genes
df_sig4_novel = df_sig4[df_sig4['replicated in nr.studies'] == 0]

# Identify unique genes
unique_genes = sorted(set(df_sig4_novel['Gene name']) - set(prior_studies['GeneSymbol']))

# Print summary
print('{} new proteins found compared to existing studies.'.format(len(unique_genes)))

In [ ]:
import ast
import itertools

studies = [i for i in df_sig4['novel'] if i!='novel']
study_occurences = []
for i in studies:
    res = ast.literal_eval(i)
    study_occurences.append(res)

study_occurences = list(itertools.chain(*study_occurences))
study_occurences_df = pd.DataFrame({'occurences':study_occurences})['occurences'].value_counts()/df_sig4.shape[0]

In [ ]:
# Count number of studies in which each gene is replicated
replication_matrix = df_sig4['replicated in nr.studies'].value_counts().sort_index()

replication_matrix_toplot = replication_matrix[replication_matrix.index<11]
replication_matrix_toplot['>10'] = replication_matrix[replication_matrix.index>10].sum()

In [ ]:
df_sig4['beta_abs']=df_sig4['beta_abs'].astype(float)

In [ ]:
df_sig4['nr.studies.q']=pd.cut(x=df_sig4['replicated in nr.studies'], bins=[-1, 5, 10, 15])

In [ ]:
df_sig4['nr.studies.q']=df_sig4['nr.studies.q'].astype(str)

In [ ]:
df_sig4.groupby('values')['replicated in nr.studies'].describe()

#### Reviewer question

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
fig.subplots_adjust(wspace=0.4)
sns.violinplot(x='nr.studies.q', y='beta_abs', data=df_sig4, hue='values', palette='Paired', ax=axes[0], cut=0)
sns.boxplot(x='values', y='replicated in nr.studies', data=df_sig4, ax=axes[1], palette='Paired')
plt.rcParams['pdf.fonttype'] = 42
plt.savefig(os.path.join(figure_path, 'reviewer_q.pdf'), dpi=120, bbox_inches='tight')

#### Fig. S7

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
fig.subplots_adjust(wspace=0.4)
axes[0].bar(x=np.arange(replication_matrix_toplot.__len__()), 
        height=replication_matrix_toplot, width=0.6)
axes[0].set_ylabel('Number of pQTLs', fontsize=14)
axes[0].set_xlabel('Number of studies', fontsize=14)
axes[0].set_xticks(np.arange(replication_matrix_toplot.__len__()), 
           labels=replication_matrix_toplot.index, 
          rotation=30);
sns.boxplot(x='values', y='replicated in nr.studies', data=df_sig4, ax=axes[1], palette='Paired')
plt.rcParams['pdf.fonttype'] = 42
plt.savefig(os.path.join(figure_path, 'FigS7.pdf'), dpi=120, bbox_inches='tight')

#### Source data

In [ ]:
sourcedata_7b = pd.DataFrame(replication_matrix_toplot).reset_index()
sourcedata_7b.columns = ['Nr. studies', 'Nr. pQTLs']
sourcedata_7b.to_excel(os.path.join(Base, 'analysis/tables/SourceData_ExtendedDataFigure6a.xlsx'))

#### Prepare top cis-pQTLs for MR

In [ ]:
cols_tokeep = ['chr', 'rs', 'rsID', 'ps', 'p_wald', '-log10(p-wald)', 'allele1', 'allele0', 'af', 'beta', 'se', 'phenotype']
df_cis = df_final[df_final['values']=='cis'].sort_values(by='-log10(p-wald)', ascending=False).drop_duplicates(subset=['phenotype'], keep='first')[cols_tokeep]

In [ ]:
df_cis=df_cis.rename({'chr':'chromosome', 'rs':'snp', 'ps':'pos', 'allele1':'alt','allele0':'ref',},axis=1)
df_cis.to_csv(os.path.join(result_path, 'HOLBAEK_top_cis_pqtls_build37.txt'), sep='\t', index=False)

#### Write to results

In [ ]:
vcf_path ='/home/projects/cpr_man/people/lilniu/projects/target/data/data_genotype/subset/Formatted/plink_qc4'
with open (os.path.join(vcf_path, 'novel_pqtl_id.txt'), 'w') as f:
    for line in df_sig4_novel['rs'].tolist():
        f.write(line + '\n')

df_sig4_novel.to_csv(os.path.join(vcf_path, 'novel_pqtl.csv'), index=False)

#### How many of the novel pQTLs were replicated in 1k? 

In [ ]:
# Extract pQTLs replicated in the replication cohort
df_replicated_1k = df_rep_1k[df_rep_1k['replicated']!='no']

# Extract the relevant columns from df_replicated_ald and df_sig4_novel
replicated_ids = df_replicated_1k['rs']+df_replicated_1k['Protein ID']
novel_ids = df_sig4_novel['rs']+df_sig4_novel['Protein ID']

# Compute the intersection of the two sets and count the number of elements
num_common_ids = len(intersection(replicated_ids, novel_ids))

# Print the result
print('The number of common IDs between the two datasets is: {}'.format(num_common_ids))
print('{}% of novel pQTLs were replicated'.format(num_common_ids/len(novel_ids)*100))

#### Prepare Supplementary Table 5

In [ ]:
df_table5 = df_sig4.copy()
#df_table5 = df_table5.reset_index().drop(['index', 'ind'], axis=1)
df_table5['study-wide significant']=np.where(df_table5['p_wald']<5e-8/420, 'yes', 'no')
df_table5['variant annotation'] = df_table5['rs'].map(dict_vep)
df_table5['novel'] = df_table5['novel'].replace('[]', 'novel')

cols_tokeep = ['pqtl_id', 'rs', 'rsID',  'chr', 'ps', 'located_gene', 'lead_located_gene', 'variant annotation', 'REF', 'ALT', 'maf', 
               'p_wald', '-log10(p-wald)', 'beta', 'se', 'study-wide significant', 
              'values', 'Protein ID', 'Gene name', 'UniParc ID','chr_protein',
              'Gene start (bp)', 'Gene end (bp)', 'Transcription start site (TSS)', 
               'nr.peptides.sig.spec', 'all.same.direction', 'tier','RNA tissue specificity', 'RNA tissue specific nTPM',
               'novel', 'replicated in nr.studies', ]

new_names = ['pQTL_ID', 'SNP', 'rsID', 'chr', 'pos', 'located_gene', 'lead_located_gene', 'VEP annotation','reference_allele', 'alternative_allele', 
            'MAF', 'p_wald', '-log10(p-wald)', 'beta', 'standard error', 'study-wide significant', 
            'cis/trans', 'Protein ID', 'Gene name', 'UniParc ID','chr_protein', 'Gene start (bp)', 
             'Gene end (bp)', 'Transcription start site (TSS)', 
             'nr.peptides.sig.spec', 'all.same.direction', 'tier','RNA tissue specificity', 'RNA tissue specific nTPM',
             'novel', 'replicated in nr.studies']

df_table5 = df_table5[cols_tokeep].rename(dict(zip(cols_tokeep, new_names)), axis=1)

In [ ]:
log2fc = pd.read_excel(os.path.join(vcf_path, 'gemma/analysis/tables/log2fc.xlsx'))
cols_tokeep = ['pQTL_ID',
               'Mean intensity (log2) 0/0', 
               'Mean intensity (log2) 0/1', 
               'Mean intensity (log2) 1/1',
               'Fold change 0/1 vs. 0/0', 
               'Fold change 1/1 vs. 0/0', 
               'AF']
df_table5_formatted = df_table5.merge(log2fc[cols_tokeep], on='pQTL_ID',how='left').drop('pQTL_ID', axis=1)

### Mapping pQTLs to GWAS Catalogue

In [ ]:
gwas_catal = pd.read_csv(os.path.join(crossref_path, 'gwas_catalog_v1.0.2-associations_e107_r2022-07-30.tsv'), 
                         sep='\t', low_memory=False)

In [ ]:
traits = [i for i in gwas_catal['DISEASE/TRAIT'].unique() if 'heart' in i.lower()]

In [ ]:
snps=gwas_catal[gwas_catal['DISEASE/TRAIT'].isin(traits)]['SNPS'].unique()

In [ ]:
# Write all mapped GWAS traits for inspection
with open(os.path.join(crossref_path, 'gwas_traits'), 'w') as file:
    for i in gwas_catal['MAPPED_TRAIT'].unique():
        file.write(str(i)+'\n')

In [ ]:
from src.config import annotate_with_gwas

In [ ]:
df_sig4_gwas = annotate_with_gwas(df_sig4, gwas_catal)

In [ ]:
overlap = set(df_sig4_gwas['rsID']) & set(snps)

In [ ]:
df_sig4_gwas[df_sig4_gwas['rsID'].isin(overlap)]['phenotype'].unique()
df_sig4_gwas[df_sig4_gwas['rsID'].isin(overlap)][['phenotype', 'lead_located_gene', 'vep', 'novel',
                                                  'Nr. GWAS_records', 'GWAS_traits', 'Nr. GWAS_traits', ]]

#### Prepare Supplementary Table 10

In [ ]:
cols_tokeep = ['pqtl_id', 'rsID', 'Protein ID', 'Gene name','lead_located_gene', 'Nr. GWAS_records','replicated in nr.studies',
                'GWAS_traits', 'Nr. GWAS_traits', 'Study accessions', 'Mapped trait urls', 'values']

df = df_sig4_gwas[cols_tokeep]
df = df[df['Nr. GWAS_traits']>0].rename({'pqtl_id':'pQTL_ID'}, axis=1)
df['novel pQTL'] = np.where(df['replicated in nr.studies']>0, 'no', 'yes')
df=df.drop(['replicated in nr.studies'], axis=1).sort_values(by='Nr. GWAS_traits', ascending=False)
df_table10=df.copy()

In [ ]:
# Write to supplementary tables
with pd.ExcelWriter(os.path.join(Base,'analysis/tables/TableS5_6_7_10_13_14.xlsx')) as writer:
    df_table5_formatted.to_excel(writer,sheet_name='ST5', index=False)
    df_cojo_formatted.to_excel(writer, sheet_name='ST6', index=False)
    df_export.to_excel(writer, sheet_name='ST7', index=True)
    df_table10.to_excel(writer, sheet_name='ST10', index=False)
    df_table13.to_excel(writer, sheet_name='ST13', index=False)
    df_table14.to_excel(writer, sheet_name='ST14', index=False)